# Entrenamos nuestra primera GRU

Ejecutamos IPPO con memoria y un presupuesto pequeño para CPU. Comprobamos que los pesos
cambian, guardamos el modelo y observamos episodios antes y después. Esta ejecución es
exploratoria: una seed y pocos pasos no establecen H1 ni H2. En este bloque usamos
rho=0; el autojuego contra versiones congeladas queda pendiente.

Consultamos [episodios y aprendizaje](../docs/EPISODIOS.md) para compartir los resultados.

In [ ]:
import json
from pathlib import Path

from darkforest.config import load_config
from train.ippo import train
from train.logging_utils import run_dir

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()
)


config_name = "aprendizaje_dos"
cfg = load_config(ROOT / "configs" / f"{config_name}.yaml")
directory = run_dir(cfg, config_name)
if not directory.is_absolute():
    directory = ROOT / directory
print("Civilizaciones:", cfg.env.n_civs)
print("Entornos simultáneos:", cfg.train.n_envs)
print("Pasos de entorno:", cfg.train.total_steps)
print(
    "Actualizaciones PPO:",
    cfg.train.total_steps // (cfg.train.n_envs * cfg.train.rollout_len) * cfg.train.update_epochs,
)

## Ejecutamos o consultamos una corrida

La siguiente celda entrena si no existe el checkpoint. Si ya terminamos esa corrida,
consultamos sus archivos sin entrenar otra vez. Una corrida incompleta produce un aviso:
podemos inspeccionarla y usar otro nombre de experimento, sin borrar ni sobrescribirla.

Recogemos fragmentos de 16 pasos; la memoria continúa dentro del mismo episodio.
El actor propone acciones y el crítico predice retornos. GAE calcula ventajas y PPO
actualiza los parámetros. Reiniciamos memoria al terminar el episodio o morir un agente.

In [ ]:
from dataclasses import replace

checkpoint = directory / "checkpoints" / "final.msgpack"
if checkpoint.exists():
    print("Consultamos el checkpoint existente:", checkpoint)
elif directory.exists():
    raise FileExistsError(f"Revisamos la corrida incompleta antes de continuar: {directory}")
else:
    local_cfg = replace(cfg, logging=replace(cfg.logging, out_dir=str(ROOT / cfg.logging.out_dir)))
    trained_state = train(local_cfg, config_name)
    print("Guardamos el checkpoint:", checkpoint)

## Comparamos los registros

La norma del cambio de parámetros debe ser positiva. El retorno puede subir o bajar
en este ejemplo corto: no interpretamos una única comparación como evidencia de mejora.

In [ ]:
report = json.loads((directory / "evaluation.json").read_text(encoding="utf-8"))
assert report["parameter_delta_norm"] > 0
report

In [ ]:
import matplotlib.pyplot as plt

history = [
    json.loads(line)
    for line in (directory / "training.jsonl").read_text(encoding="utf-8").splitlines()
]
fig, axes = plt.subplots(1, 2, figsize=(9, 3))
axes[0].plot([row["step"] for row in history], [row["loss"] for row in history])
axes[0].set(xlabel="Pasos de entorno", ylabel="Pérdida PPO")
axes[1].plot([row["step"] for row in history], [row["entropy"] for row in history])
axes[1].set(xlabel="Pasos de entorno", ylabel="Entropía del actor")
fig.tight_layout()
plt.show()

## Leemos episodios guardados

Cargamos arrays y calculamos métricas sin ejecutar el entorno ni actualizar pesos.
Reproducimos la animación desde [demo.ipynb](demo.ipynb).

In [ ]:
from analysis.metrics import summarize
from darkforest.trajectories import load_episode

for episode_id in (1, 2):
    episode_path = directory / "trajectories" / f"episode_{episode_id:06d}.npz"
    trajectory, episode_metadata = load_episode(episode_path)
    print(episode_metadata["policy"], summarize(trajectory))

## Continuamos con cuatro civilizaciones

Cambiamos config_name a aprendizaje_cuatro, reiniciamos el kernel y ejecutamos las celdas.
Entrenamos una política compartida nueva para esa configuración; no suponemos que un modelo
entrenado con dos civilizaciones ya generaliza a cuatro. Dejamos seis y ocho para los
experimentos de escala y coaliciones.

Compartimos la carpeta de corrida completa por el medio acordado por el equipo.
Quienes analizan o visualizan pueden leer esos archivos sin repetir el entrenamiento.
Limpiamos las salidas del notebook antes de versionarlo.